In [18]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import shutil

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data_raw"
META_DIR = PROJECT_DIR / "metadata"
OUT_DIR = PROJECT_DIR / "outputs"

WEGE_PATH = PROJECT_DIR / "data_raw" / "MiD2023_Wege.csv"
AUTOS_PATH = PROJECT_DIR / "data_raw" / "MiD2023_Autos.csv"

OUT_DIR.mkdir(exist_ok=True)

OUT_PATH = PROJECT_DIR / "data_processed" / "wege_auto_linkage_core.csv"

b1_file = META_DIR / "MiD2023_Codepläne_B1_Standard_v1.1.xlsx"
out_file = OUT_DIR / "B1_Standard_with_common_columns.xlsx"

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")
print("B1 file exists:", b1_file.exists())

Found 7 CSV files
B1 file exists: True


In [19]:
cols_by_file = {}

for file in csv_files:
    df0 = pd.read_csv(
        file,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
        nrows=0
    )
    cols_by_file[file.name] = list(df0.columns)

for fname, cols in cols_by_file.items():
    print(fname, ":", len(cols), "columns")

MiD2023_Autos.csv : 77 columns
MiD2023_Etappen.csv : 120 columns
MiD2023_Haushalte.csv : 79 columns
MiD2023_Personen.csv : 261 columns
MiD2023_Reisen.csv : 97 columns
MiD2023_Tagesreisen.csv : 83 columns
MiD2023_Wege.csv : 209 columns


In [20]:
col_to_files = defaultdict(list)

for fname, cols in cols_by_file.items():
    for col in cols:
        col_to_files[col].append(fname)

common_df = pd.DataFrame([
    {
        "column": col,
        "appears_in_n_files": len(files),
        "files": " | ".join(files)
    }
    for col, files in col_to_files.items()
    if len(files) >= 2
]).sort_values(["appears_in_n_files", "column"], ascending=[False, True])

common_df.head(10)

,column,appears_in_n_files,files
2,BASISAUF,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
23,BLAND,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
24,BLAND_GEO,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
16,H_ANZAUTO,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
14,H_GR,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
0,H_ID,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
1,MODE,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
32,MSIndex,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
25,RegioStaR17,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
28,RegioStaR2,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...


In [21]:
b1_sheets = pd.read_excel(
    b1_file,
    sheet_name=None,
    dtype=str,
    header=1
)

for sheet_name, df in b1_sheets.items():
    print(sheet_name, df.columns.tolist())


Haushalte ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Personen ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Wege ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Autos ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Reisen ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Tagesreisen ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']
Etappen ['Position', 'Variable', 'Variablenlabel', 'Wert', 'Wertelabel', 'Messniveau', 'Format']


In [22]:
label_records = []

for sheet_name, df in b1_sheets.items():
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    
    if "Variable" not in df.columns or "Variablenlabel" not in df.columns:
        print(f"Missing expected columns in sheet: {sheet_name}")
        print(df.columns.tolist())
        continue
    
    temp = df[["Variable", "Variablenlabel"]].copy()
    temp = temp.dropna(subset=["Variable"])
    
    temp["column"] = temp["Variable"].astype(str).str.strip()
    temp["Variablenlabel"] = temp["Variablenlabel"].astype(str).str.strip()
    temp["source_sheet"] = sheet_name
    
    temp = temp[["column", "Variablenlabel", "source_sheet"]]
    label_records.append(temp)

labels_df = pd.concat(label_records, ignore_index=True)

labels_df.head(10)

,column,Variablenlabel,source_sheet
0,H_ID,Haushalts-ID,Haushalte
1,H_GEW,Gewichtungsfaktor Haushalte,Haushalte
2,H_HOCH,Hochrechnungsfaktor Haushalte,Haushalte
3,MODE,Erhebungsmethode,Haushalte
4,BASISAUF,Basis- oder Aufstockungsstichprobe,Haushalte
5,TEILSTP,Teilstichprobe,Haushalte
6,M_CAR,Modul - Fahrzeugmerkmale und Fahrzeugbesitz,Haushalte
7,H_ART,Art des Haushalts,Haushalte
8,H_GR,Haushaltsgröße,Haushalte
9,hhgr_gr,Haushaltsgröße in Gruppen (1 bis 5+),Haushalte


In [23]:
labels_grouped = (
    labels_df
    .groupby("column", as_index=False)
    .agg({
        "Variablenlabel": lambda x: " | ".join(sorted(set(v for v in x if v and v != "nan"))),
        "source_sheet": lambda x: " | ".join(sorted(set(x)))
    })
)

common_labeled = common_df.merge(
    labels_grouped,
    on="column",
    how="left"
)

common_labeled = common_labeled[
    ["column", "Variablenlabel", "source_sheet", "appears_in_n_files", "files"]
]

print("Number of repeated column names:", len(common_labeled))
print("Shape:", common_labeled.shape)
common_labeled.head(5)

Number of repeated column names: 124
Shape: (124, 5)


,column,Variablenlabel,source_sheet,appears_in_n_files,files
0,BASISAUF,Basis- oder Aufstockungsstichprobe,Autos | Etappen | Haushalte | Personen | Reise...,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
1,BLAND,Bundesland,Autos | Etappen | Haushalte | Personen | Reise...,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
2,BLAND_GEO,Bundesland nach Himmelsrichtung,Autos | Etappen | Haushalte | Personen | Reise...,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
3,H_ANZAUTO,Anzahl Autos im HH,Autos | Etappen | Haushalte | Personen | Reise...,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...
4,H_GR,Haushaltsgröße,Autos | Etappen | Haushalte | Personen | Reise...,7,MiD2023_Autos.csv | MiD2023_Etappen.csv | MiD2...


In [24]:

# out_file = OUT_DIR / "B1_Standard_with_common_columns.xlsx"

# shutil.copy2(b1_file, out_file)

# with pd.ExcelWriter(
#     out_file,
#     engine="openpyxl",
#     mode="a",
#     if_sheet_exists="replace"
# ) as writer:
#     common_labeled.to_excel(
#         writer,
#         sheet_name="Common_columns",
#         index=False
#     )

# print("Saved to:", out_file)

In [25]:
def read_mid_csv(file, nrows=None, dtype=str, **read_csv_kwargs):
    sep = ","

    return pd.read_csv(
        file,
        sep=sep,
        encoding="utf-8-sig",
        nrows=nrows,
        dtype=dtype,
        low_memory=False,
        **read_csv_kwargs,
    )


In [26]:
# def count_rows_by_chunks(file, chunksize=200_000):
#     total = 0
#     sep = ","
    
#     for chunk in pd.read_csv(
#         file,
#         sep=sep,
#         encoding="utf-8-sig",
#         dtype=str,
#         chunksize=chunksize
#     ):
#         total += len(chunk)
    
#     return total

# audit_records = {}
# data_samples = {}

# for file in csv_files:
#     df_head = read_mid_csv(file, nrows=5)
#     n_rows = count_rows_by_chunks(file)
    
#     audit_records[file.name] = {
#         "file": file.name,
#         "n_rows": n_rows,
#         "n_columns": df_head.shape[1],
#         "sep": ",",
#         "columns": " | ".join(df_head.columns)
#     }
    
#     data_samples[file.name] = df_head
    
#     print(file.name, "rows:", n_rows, "columns:", df_head.shape[1])
#     display(df_head)

In [ ]:
# Shared linkage implementation: union of both analyses' fields and available labels.
import sys

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from scripts.link_wege_autos_core import (
    AUTO_COLS,
    LINKAGE_DISPLAY_COLS,
    WEGE_COLS,
    build_trip_vehicle_linkage,
)


In [28]:
def trip_veh_linkage():
    # Broad QA/enrichment view: no multi-car or driver restriction.
    wege, _, _, linked = build_trip_vehicle_linkage(
        WEGE_PATH, AUTOS_PATH, reader=read_mid_csv
    )

    linkage_summary = pd.DataFrame({
        "metric": [
            "all Wege rows read",
            "linkable Wege rows (W_WAUTO in 1/2/3)",
            "matched rows",
            "unmatched rows",
            "unique linked vehicles",
        ],
        "value": [
            len(wege),
            len(linked),
            int(linked["link_status"].eq("matched").sum()),
            int(linked["link_status"].eq("unmatched").sum()),
            linked.loc[linked["link_status"].eq("matched"), ["H_ID", "A_ID"]].drop_duplicates().shape[0],
        ],
    })

    linked = linked[LINKAGE_DISPLAY_COLS].copy()

    return linked, linkage_summary


linked_wege_autos, linkage_summary = trip_veh_linkage()

display(linkage_summary)
display(linked_wege_autos.head(20))


,metric,value
0,all Wege rows read,1087393
1,linkable Wege rows (W_WAUTO in 1/2/3),291426
2,matched rows,291426
3,unmatched rows,0
4,unique linked vehicles,80983


,H_ID,P_ID,W_ID,W_WAUTO,A_ID,n_cars_hh,link_status,W_FMF,W_FMF_label,zweck,...,antrieb_label,A_ANTRIEB,A_BAUJ,A_ERWJ,A_HALTER,A_HALTER_label,seg_kba,seg_kba_gr,status,A_JAHRESFL
0,10000010,1,1,1,1,1,matched,1,Fahrer/in,5,...,Diesel,2,2012,2014,1,Privatwagen,10,2,1,10000
1,10000010,1,2,1,1,1,matched,1,Fahrer/in,4,...,Diesel,2,2012,2014,1,Privatwagen,10,2,1,10000
2,10000480,1,1,2,2,2,matched,2,Mitfahrer/in,7,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
3,10000480,1,2,2,2,2,matched,2,Mitfahrer/in,8,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
4,10000480,1,3,2,2,2,matched,2,Mitfahrer/in,8,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
5,10000480,2,1,2,2,2,matched,1,Fahrer/in,7,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
6,10000480,2,2,2,2,2,matched,1,Fahrer/in,7,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
7,10000480,2,3,2,2,2,matched,1,Fahrer/in,8,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
8,10000480,3,1,2,2,2,matched,403,Person unter 16 Jahren,10,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000
9,10000480,3,2,2,2,2,matched,403,Person unter 16 Jahren,10,...,Benzin,1,2020,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,3,30000


In [29]:
# Field selection and linkage now come from the shared builder above.
VEHICLE_ATTRS_FOR_DIFFERENCE = ["antrieb", "A_BAUJ", "A_HALTER", "seg_kba", "status"]


In [30]:
def check_vehicle_allocation_support():
    # Allocation-specific filters begin only after the shared all-household linkage.
    _, _, car_counts, linked_full = build_trip_vehicle_linkage(
        WEGE_PATH, AUTOS_PATH, reader=read_mid_csv
    )
    linked_full["is_driver_trip"] = linked_full["W_FMF"].isin(["1", "3"])
    linked_full["wegkm_imp_num"] = pd.to_numeric(linked_full["wegkm_imp"], errors="coerce")
    linked_full["wegmin_imp2_num"] = pd.to_numeric(linked_full["wegmin_imp2"], errors="coerce")

    # Treat "linked" as an actual successful Wege-to-Autos match.
    linked_trips = linked_full[linked_full["link_status"].eq("matched")].copy()
    multi_car_linked_trips = linked_trips[linked_trips["n_cars_hh"].ge(2)].copy()
    driver_trips = multi_car_linked_trips[multi_car_linked_trips["is_driver_trip"]].copy()
    driver_trips["hh_trip_key"] = driver_trips["H_ID"].astype(str) + "|" + driver_trips["W_ID"].astype(str)

    hh_driver_support = driver_trips.groupby("H_ID").agg(
        n_cars_hh=("n_cars_hh", "first"),
        n_driver_trips=("hh_trip_key", "nunique"),
        n_driver_person_trips=("W_ID", "size"),
        n_used_cars=("A_ID", "nunique"),
        n_persons=("P_ID", "nunique"),
        n_purposes=("zweck", "nunique"),
        n_distance_values=("wegkm_imp", "nunique"),
        median_distance_km=("wegkm_imp_num", "median"),
    )

    used_vehicles = driver_trips.drop_duplicates(["H_ID", "A_ID"])
    attr_variation = (
        used_vehicles
        .groupby("H_ID")[VEHICLE_ATTRS_FOR_DIFFERENCE]
        .nunique(dropna=False)
        .ge(2)
    )
    hh_driver_support["vehicle_attrs_differ"] = attr_variation.any(axis=1)
    hh_driver_support = hh_driver_support.reset_index()

    hh_driver_support["supports_allocation_pattern"] = (
        hh_driver_support["n_driver_trips"].ge(2)
        & hh_driver_support["n_used_cars"].ge(2)
        & hh_driver_support["vehicle_attrs_differ"]
        & hh_driver_support["n_purposes"].ge(2)
    )

    def step_record(step, hh_ids):
        hh_ids = set(hh_ids)
        linked_in_step = multi_car_linked_trips[multi_car_linked_trips["H_ID"].isin(hh_ids)]
        return {
            "step": step,
            "n_households": len(hh_ids),
            "n_linkable_car_trips": int(linked_in_step.shape[0]),
            "n_driver_trips": int(
                linked_in_step.loc[linked_in_step["is_driver_trip"], ["H_ID", "W_ID"]]
                .drop_duplicates()
                .shape[0]
            ),
            "n_driver_person_trips": int(linked_in_step["is_driver_trip"].sum()),
        }

    multi_car_hh = set(car_counts.loc[car_counts["n_cars_hh"].ge(2), "H_ID"])

    hh_2plus_driver = set(hh_driver_support.loc[hh_driver_support["n_driver_trips"].ge(2), "H_ID"])
    hh_2plus_used_cars = set(hh_driver_support.loc[
        hh_driver_support["n_driver_trips"].ge(2) & hh_driver_support["n_used_cars"].ge(2), "H_ID"
    ])
    hh_attrs_differ = set(hh_driver_support.loc[
        hh_driver_support["n_driver_trips"].ge(2)
        & hh_driver_support["n_used_cars"].ge(2)
        & hh_driver_support["vehicle_attrs_differ"], "H_ID"
    ])
    hh_2plus_purposes = set(hh_driver_support.loc[
        hh_driver_support["supports_allocation_pattern"], "H_ID"
    ])
    hh_4plus_driver = set(hh_driver_support.loc[
        hh_driver_support["supports_allocation_pattern"] & hh_driver_support["n_driver_trips"].ge(4), "H_ID"
    ])

    allocation_support_funnel = pd.DataFrame([
        step_record("HH with at least 2 cars in Autos", multi_car_hh),
        step_record("+ at least 2 linked driver-trips", hh_2plus_driver),
        step_record("+ at least 2 different cars observed", hh_2plus_used_cars),
        step_record("+ observed cars differ in selected attributes", hh_attrs_differ),
        step_record("+ at least 2 trip purposes", hh_2plus_purposes),
        step_record("+ at least 4 driver-trips", hh_4plus_driver),
    ])[
        ["step", "n_households", "n_linkable_car_trips", "n_driver_person_trips", "n_driver_trips"]
    ]

    supported_hh = set(hh_driver_support.loc[hh_driver_support["supports_allocation_pattern"], "H_ID"])
    allocation_driver_trips_person = driver_trips[driver_trips["H_ID"].isin(supported_hh)].copy()

    def join_unique(values):
        values = pd.Series(values).dropna().astype(str)
        values = values[values.ne("")]
        return "; ".join(pd.unique(values))

    allocation_driver_trips_person = allocation_driver_trips_person.sort_values(["H_ID", "W_ID", "P_ID", "A_ID"])
    hh_trip_counts = allocation_driver_trips_person.groupby(["H_ID", "W_ID"], as_index=False).agg(
        n_driver_person_trips=("W_ID", "size"),
        n_driver_persons=("P_ID", "nunique"),
        n_used_cars_in_hh_trip=("A_ID", "nunique"),
    )
    hh_trip_lists = allocation_driver_trips_person.groupby(["H_ID", "W_ID"], as_index=False).agg(
        P_ID=("P_ID", join_unique),
        W_WAUTO=("W_WAUTO", join_unique),
        A_ID=("A_ID", join_unique),
        W_FMF=("W_FMF", join_unique),
        W_FMF_label=("W_FMF_label", join_unique),
        zweck=("zweck", join_unique),
        W_ZWECK=("W_ZWECK", join_unique),
    )
    representative_cols = [
        col for col in LINKAGE_DISPLAY_COLS
        if col not in ["P_ID", "W_WAUTO", "A_ID", "W_FMF", "W_FMF_label", "zweck", "W_ZWECK"]
    ]
    representative_rows = allocation_driver_trips_person.drop_duplicates(["H_ID", "W_ID"])[representative_cols]
    allocation_driver_trips = (
        hh_trip_counts
        .merge(hh_trip_lists, on=["H_ID", "W_ID"], how="left", validate="one_to_one")
        .merge(representative_rows, on=["H_ID", "W_ID"], how="left", validate="one_to_one")
    )

    allocation_driver_trips = allocation_driver_trips[
        ["H_ID", "W_ID", "n_driver_person_trips", "n_driver_persons", "n_used_cars_in_hh_trip"]
        + [col for col in LINKAGE_DISPLAY_COLS if col not in ["H_ID", "W_ID"]]
    ].copy()

    return linked_full, hh_driver_support, allocation_support_funnel, allocation_driver_trips


In [31]:
linked_full, hh_driver_support, allocation_support_funnel, allocation_driver_trips = check_vehicle_allocation_support()

multi_car_linked = linked_full[
    linked_full["link_status"].eq("matched") & linked_full["n_cars_hh"].ge(2)
].copy()

n_total_linkable = len(multi_car_linked)
n_total_linkable_hh = len(multi_car_linked[["H_ID", "W_ID"]].drop_duplicates())
n_driver_person = multi_car_linked["is_driver_trip"].sum()
n_driver_hh = len(
    multi_car_linked.loc[multi_car_linked["is_driver_trip"], ["H_ID", "W_ID"]].drop_duplicates()
)
n_non_driver = n_total_linkable - n_driver_person

print("Multi-car household linkable car person-trip rows:", n_total_linkable)
print("Multi-car household linkable car trips (HH-W_ID level):", n_total_linkable_hh)
print("Driver trips (HH-W_ID level):", n_driver_hh)
print("Driver person-trip rows:", n_driver_person)
print("Non-driver household-car person-trip rows:", n_non_driver)
print("Non-driver share:", n_non_driver / n_total_linkable if n_total_linkable else None)

KeyboardInterrupt: 

In [ ]:
if "allocation_support_funnel" not in globals():
    linked_full, hh_driver_support, allocation_support_funnel, allocation_driver_trips = check_vehicle_allocation_support()

display(allocation_support_funnel)
display(
    hh_driver_support
    .sort_values(["supports_allocation_pattern", "n_driver_trips", "n_used_cars"], ascending=False)
    .head(20)
)
display(allocation_driver_trips.head(30))


,step,n_households,n_linkable_car_trips,n_driver_person_trips,n_driver_trips
0,HH with at least 2 cars in Autos,85408,184661,140258,114736
1,+ at least 2 linked driver-trips,32596,176141,135717,110499
2,+ at least 2 different cars observed,11853,89049,71852,48632
3,+ observed cars differ in selected attributes,11754,88389,71304,48260
4,+ at least 2 trip purposes,11694,88152,71132,48119
5,+ at least 4 driver-trips,7246,66698,52775,37336


,H_ID,n_cars_hh,n_driver_trips,n_driver_person_trips,n_used_cars,n_persons,n_purposes,n_distance_values,median_distance_km,vehicle_attrs_differ,supports_allocation_pattern
1954,11082170,3,12,23,3,3,7,12,5.700,True,True
7237,13919590,3,12,15,3,2,5,5,2.850,True,True
13142,17400590,3,12,24,3,2,4,13,15.200,True,True
1523,10837680,2,12,14,2,2,5,5,2.850,True,True
3978,12154760,2,12,16,2,2,4,5,8.550,True,True
7266,13932750,2,12,14,2,2,5,8,3.800,True,True
9921,15398770,2,12,14,2,2,5,7,15.200,True,True
10583,15749140,2,12,17,2,2,6,6,11.400,True,True
11793,16401280,2,12,15,2,2,5,12,8.460,True,True
13503,17575780,3,12,12,2,1,6,8,7.125,True,True


,H_ID,W_ID,n_driver_person_trips,n_driver_persons,n_used_cars_in_hh_trip,P_ID,W_WAUTO,A_ID,n_cars_hh,link_status,...,antrieb_label,A_ANTRIEB,A_BAUJ,A_ERWJ,A_HALTER,A_HALTER_label,seg_kba,seg_kba_gr,status,A_JAHRESFL
0,10000600,1,1,1,1,2,2,2,2,matched,...,Diesel,2,2007,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,11,3,1,9000
1,10000600,2,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
2,10000600,3,1,1,1,1,1,1,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
3,10000600,4,1,1,1,1,1,1,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
4,10000600,5,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
5,10000600,6,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
6,10000600,7,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,4,3,2,25000
7,10000600,8,1,1,1,2,2,2,2,matched,...,Diesel,2,2007,10101,101,Modul Fahrzeugmerkmale und Fahrzeugbesitz nich...,11,3,1,9000
8,10001310,1,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,2017,1,Privatwagen,12,3,2,20000
9,10001310,2,2,2,2,1; 2,1; 2,1; 2,2,matched,...,Diesel,2,2017,2017,1,Privatwagen,12,3,2,20000


In [ ]:
allocation_driver_trips.shape

(48119, 29)

In [ ]:
hh_driver_support.shape

(36833, 11)

In [ ]:
hh = pd.read_csv(
    DATA_PATH,
    sep=None,         
    engine="python",
    dtype=str,         
    encoding="utf-8-sig"
)